# Gemma 4 31B — AL Quant Inference (Google Colab)

Downloads **private** Oxidize AL-family GGUF quants from Hugging Face (`freakyskittle/gemma-4-31B-it-AL-GGUF`) and runs generation with `llama-cpp-python`.

**GPU recommended.** AL5 needs ~18GB VRAM; use `AL5_XS` on smaller GPUs.

**Auth:** Colab → Secrets → add `HF_TOKEN` (read access to your private repo).

In [ ]:
import sys
get_ipython().system("{sys.executable} -m pip install -q huggingface_hub llama-cpp-python")

HF_REPO = "freakyskittle/gemma-4-31B-it-AL-GGUF"
QUANT = "AL5"  # AL5 | AL6 | AL8 | AL5_XS
FILENAME = f"gemma-4-31B-it-{QUANT}.gguf"

In [ ]:
from huggingface_hub import hf_hub_download
from pathlib import Path

# Private repo — authenticate in Colab: Secrets → HF_TOKEN, or run `huggingface-cli login`
import os
token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_HUB_TOKEN")

MODEL_PATH = Path(
    hf_hub_download(
        repo_id=HF_REPO,
        filename=FILENAME,
        token=token,
    )
)
print(f"model: {MODEL_PATH} ({MODEL_PATH.stat().st_size / 1e9:.2f} GB)")

In [ ]:
from llama_cpp import Llama

llm = Llama(
    model_path=str(MODEL_PATH),
    n_gpu_layers=-1,
    n_ctx=4096,
    verbose=False,
)

prompt = (
    "<|turn>user\nExplain quantum computing in simple terms.<turn|>\n"
    "<|turn>model\n<|channel>final<|message|>"
)
out = llm(prompt, max_tokens=256, temperature=0.7)
print(out["choices"][0]["text"])